[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-02-ollama-local-setup.ipynb#scrollTo=a1b2c3d4)

---
# Day 2 · Running Llama Locally with Ollama
**certified-journeys / llama-certified** · Day 2 · Local Inference

> **Goal for today:** By the end of this notebook you can install Ollama, pull and run Llama 3.2, write a custom Modelfile with a system prompt and parameters, and call the Ollama REST API from Python.


In [ ]:
%pip install -q requests


## Step 1 · Installing Ollama

Ollama is a single-binary runtime that manages model weights (GGUF files), exposes a REST API on `localhost:11434`, and provides a CLI for pulling, running, and creating models.

### Installation (run once on your machine — not inside Colab)

| Platform | Command |
|---|---|
| macOS | `brew install ollama` or download the .dmg from ollama.com/download |
| Linux | `curl -fsSL https://ollama.com/install.sh \| sh` |
| Windows | Download the installer from [ollama.com/download](https://ollama.com/download) |

After install, the Ollama server starts automatically. Verify it is running:

```bash
ollama --version
curl http://localhost:11434
```

You should see `Ollama is running` in the curl response.


In [ ]:
# In Colab we cannot run a local Ollama server, so we mock the responses.
# On your local machine, replace MOCK_MODE = True with MOCK_MODE = False
# and uncomment the real requests below.

import json
import requests
from datetime import datetime

MOCK_MODE = True          # set False when running against a real Ollama server
OLLAMA_BASE = 'http://localhost:11434'

def ollama_get(path):
    """GET wrapper — returns parsed JSON or raises."""
    if MOCK_MODE:
        return {'mock': True, 'path': path}
    resp = requests.get(f'{OLLAMA_BASE}{path}', timeout=10)
    resp.raise_for_status()
    return resp.json()

# Check that the server is alive
health = ollama_get('/')
print('Server health:', health)


### What just happened?

- **`MOCK_MODE = True`** lets this notebook run end-to-end in Colab without a real server. Flip it to `False` on your local machine.
- **`http://localhost:11434`** is the default Ollama bind address — configurable via the `OLLAMA_HOST` environment variable.
- A `GET /` response of `Ollama is running` confirms the daemon is up before you make model calls.


## Step 2 · Pulling and Running a Model

Ollama downloads models from the [Ollama library](https://ollama.com/library). Each model tag maps to a specific GGUF quantization.

```bash
# Pull the 3-billion-parameter Llama 3.2 (Q4_K_M, ~2 GB)
ollama pull llama3.2:3b

# Interactive chat in the terminal
ollama run llama3.2:3b
```

### Key CLI commands

| Command | Purpose |
|---|---|
| `ollama pull <model>` | Download a model |
| `ollama run <model>` | Interactive REPL chat |
| `ollama list` | List downloaded models |
| `ollama rm <model>` | Delete a model |
| `ollama create <name> -f Modelfile` | Build a custom model |
| `ollama show <model>` | Inspect model metadata |


In [ ]:
# List available models via the REST API
# Real endpoint: GET /api/tags

MOCK_MODELS = {
    'models': [
        {
            'name': 'llama3.2:3b',
            'modified_at': '2024-10-15T10:00:00Z',
            'size': 2019399752,           # bytes — ~1.9 GB
            'details': {'family': 'llama', 'parameter_size': '3B', 'quantization_level': 'Q4_K_M'}
        }
    ]
}

def list_models():
    if MOCK_MODE:
        return MOCK_MODELS
    return requests.get(f'{OLLAMA_BASE}/api/tags', timeout=10).json()

models_data = list_models()
print(f'Downloaded models ({len(models_data["models"])} total):')
for m in models_data['models']:
    size_gb = m['size'] / 1e9
    details = m.get('details', {})
    print(f'  {m["name"]:30s}  {size_gb:.1f} GB  quant={details.get("quantization_level", "?")}')


### What just happened?

- **`GET /api/tags`** returns all locally downloaded models with size and metadata.
- **`Q4_K_M`** is Ollama's default quantization for `llama3.2:3b` — 4-bit with K-means grouping, ~1.9 GB on disk.
- **`size`** is in bytes. Dividing by `1e9` gives a human-readable GB value.


## Step 3 · Writing a Custom Modelfile

A [Modelfile](https://github.com/ollama/ollama/blob/main/docs/modelfile.md) is an Ollama-specific config that layers a system prompt, parameters, and a template on top of a base model — similar to a Dockerfile for LLMs.

### Key Modelfile instructions

| Instruction | Purpose | Example |
|---|---|---|
| `FROM` | Base model to build on | `FROM llama3.2:3b` |
| `SYSTEM` | Persistent system prompt | `SYSTEM You are a Python tutor.` |
| `PARAMETER temperature` | Sampling temperature (0–2) | `PARAMETER temperature 0.7` |
| `PARAMETER top_p` | Nucleus sampling cutoff | `PARAMETER top_p 0.9` |
| `PARAMETER num_ctx` | Context window size (tokens) | `PARAMETER num_ctx 4096` |
| `PARAMETER stop` | Custom stop sequences | `PARAMETER stop "<|end|>"` |
| `TEMPLATE` | Custom chat template | `TEMPLATE "{{ .System }}\n{{ .Prompt }}"` |


In [ ]:
import pathlib

# Write a Modelfile to disk — ollama create reads it from the filesystem.
# We also print it here for inspection.

MODELFILE_CONTENT = '''\
FROM llama3.2:3b

SYSTEM """
You are a concise Python tutor for intermediate developers.
Always respond in plain text without markdown formatting.
Keep answers under 150 words unless the user asks for detail.
"""

PARAMETER temperature 0.4
PARAMETER top_p 0.85
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
'''

mf_path = pathlib.Path('/tmp/Modelfile-python-tutor')
mf_path.write_text(MODELFILE_CONTENT)
print(f'Modelfile written to {mf_path}\n')
print(MODELFILE_CONTENT)


In [ ]:
# Create the custom model — run this in your terminal (not in Colab):
#   ollama create python-tutor -f /tmp/Modelfile-python-tutor
#
# Then start chatting:
#   ollama run python-tutor

# Here we demonstrate what the API call looks like programmatically
# via POST /api/create (streams progress events).

def mock_create_model(name, modelfile_path):
    """Simulate the streaming progress events from POST /api/create."""
    events = [
        {'status': 'reading model metadata'},
        {'status': 'creating system layer'},
        {'status': 'creating parameters layer'},
        {'status': f'success', 'name': name},
    ]
    for e in events:
        print(f'  [{e["status"]}]')
    return True

def create_model(name, modelfile_path):
    if MOCK_MODE:
        return mock_create_model(name, modelfile_path)
    payload = {'name': name, 'modelfile': pathlib.Path(modelfile_path).read_text()}
    resp = requests.post(f'{OLLAMA_BASE}/api/create', json=payload, stream=True, timeout=120)
    for line in resp.iter_lines():
        if line:
            event = json.loads(line)
            print(f'  [{event.get("status", "...")}]')
    return True

print('Creating model python-tutor from Modelfile...')
create_model('python-tutor', mf_path)
print('Done.')


### What just happened?

- **`PARAMETER temperature 0.4`** makes the model more deterministic — good for tutoring where factual accuracy matters more than creativity.
- **`PARAMETER num_ctx 4096`** sets the KV-cache window. Larger values let the model see more conversation history but use more RAM.
- **`POST /api/create` streams** line-delimited JSON events so you can show progress for large models.
- **The system prompt is baked into the model layer** — it persists across sessions without repeating it in every API call.


## Step 4 · Calling the REST API from Python

Ollama exposes two main chat endpoints:

| Endpoint | Method | Use case |
|---|---|---|
| `/api/generate` | POST | Single-turn completion (stateless) |
| `/api/chat` | POST | Multi-turn conversation with message history |
| `/api/embeddings` | POST | Generate embeddings for a prompt |

Both endpoints **stream by default**. Set `"stream": false` in the request body to receive a single complete JSON response — much simpler for scripting.

> **Tip:** The `/api/chat` endpoint is OpenAI-compatible in structure. Many libraries that support the OpenAI SDK can point at `http://localhost:11434/v1` instead.


In [ ]:
# POST /api/chat — multi-turn conversation
# stream=False returns a single JSON object instead of a stream of events.

MOCK_CHAT_RESPONSE = {
    'model': 'llama3.2:3b',
    'created_at': '2024-10-15T10:05:00Z',
    'message': {
        'role': 'assistant',
        'content': (
            'A generator is a function that yields values one at a time instead of '
            'returning a list all at once. Use it when the full result set is large '
            'or infinite — it keeps memory constant. Example: '
            'def count_up(n): [yield i for i in range(n)]'
        )
    },
    'done': True,
    'eval_count': 72,
    'eval_duration': 1_430_000_000,    # nanoseconds
    'prompt_eval_count': 31,
}

def chat(model, messages, stream=False):
    """Call POST /api/chat and return the assistant message dict."""
    if MOCK_MODE:
        return MOCK_CHAT_RESPONSE
    payload = {'model': model, 'messages': messages, 'stream': stream}
    resp = requests.post(f'{OLLAMA_BASE}/api/chat', json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()

messages = [
    {'role': 'user', 'content': 'Explain Python generators in under 100 words.'}
]

result = chat('llama3.2:3b', messages)
print('Response:')
print(result['message']['content'])
print()

# Compute tokens/sec from Ollama's timing fields
tokens = result.get('eval_count', 0)
duration_s = result.get('eval_duration', 1) / 1e9    # nanoseconds → seconds
print(f'Tokens generated : {tokens}')
print(f'Time             : {duration_s:.2f}s')
if duration_s > 0:
    print(f'Throughput       : {tokens / duration_s:.1f} tok/s')


### What just happened?

- **`stream=False`** collapses the streaming response into a single JSON object — `result['message']['content']` is the complete reply.
- **`eval_count` and `eval_duration`** in the response let you compute real tokens/sec — useful for benchmarking hardware.
- **`prompt_eval_count`** is the number of tokens in the input prompt — keep this in mind when you approach `num_ctx` limits.


## Step 5 · Multi-turn Conversation

The `/api/chat` endpoint maintains context by passing the full conversation history in each request. You accumulate messages yourself — Ollama is stateless between requests.

```
Request N:
  messages: [user1, assistant1, user2, assistant2, ..., userN]

Response N:
  message: {role: 'assistant', content: '...'}
  → append to local history, send again for turn N+1
```


In [ ]:
# Build a simple multi-turn chat loop

MOCK_TURNS = [
    'Decorators are functions that wrap other functions to add behaviour without modifying the original code.',
    'The @functools.wraps decorator copies the wrapped function name and docstring to the wrapper, preserving introspection.',
]

def multi_turn_chat(model, system_prompt, turns):
    """
    Run a list of user messages as a conversation.
    Returns the full message history.
    """
    history = []
    if system_prompt:
        history.append({'role': 'system', 'content': system_prompt})

    for idx, user_text in enumerate(turns):
        history.append({'role': 'user', 'content': user_text})

        if MOCK_MODE:
            # Simulate assistant reply from pre-canned responses
            reply = MOCK_TURNS[idx] if idx < len(MOCK_TURNS) else 'Mock response.'
        else:
            result = chat(model, history)
            reply = result['message']['content']

        history.append({'role': 'assistant', 'content': reply})
        print(f'--- Turn {idx + 1} ---')
        print(f'User     : {user_text}')
        print(f'Assistant: {reply}')
        print()

    return history

sys_prompt = 'You are a concise Python tutor. Keep every answer under 60 words.'
user_turns = [
    'What is a Python decorator?',
    'Why would I use @functools.wraps inside my decorator?',
]

history = multi_turn_chat('llama3.2:3b', sys_prompt, user_turns)
print(f'Total messages in history: {len(history)}')


### What just happened?

- **History accumulates client-side** — each request includes all prior turns so the model sees context without the server storing session state.
- **System prompts can be passed inline** in the `messages` array with `role: 'system'` — or baked into the Modelfile so you never have to send them again.
- **Turn count grows linearly** and eventually hits `num_ctx`. Summarise or trim old turns to stay within the context window for long sessions.


## Step 6 · Using the OpenAI-Compatible Endpoint

Ollama exposes an OpenAI-compatible API at `/v1/chat/completions`. Any library or tool that speaks OpenAI's protocol can target Ollama with a one-line config change.

```python
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',   # any non-empty string works
)

response = client.chat.completions.create(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Hello'}]
)
print(response.choices[0].message.content)
```

Below we replicate this with `requests` so no additional packages are needed.


In [ ]:
# OpenAI-compatible endpoint via raw requests (no openai package needed)

MOCK_OAI_RESPONSE = {
    'id': 'chatcmpl-mock001',
    'object': 'chat.completion',
    'created': 1728979200,
    'model': 'llama3.2:3b',
    'choices': [
        {
            'index': 0,
            'message': {
                'role': 'assistant',
                'content': 'Hello! How can I help you with Python today?'
            },
            'finish_reason': 'stop'
        }
    ],
    'usage': {'prompt_tokens': 12, 'completion_tokens': 11, 'total_tokens': 23}
}

def oai_chat(model, messages):
    """Call the OpenAI-compatible /v1/chat/completions endpoint."""
    if MOCK_MODE:
        return MOCK_OAI_RESPONSE
    payload = {'model': model, 'messages': messages}
    resp = requests.post(
        f'{OLLAMA_BASE}/v1/chat/completions',
        json=payload,
        headers={'Authorization': 'Bearer ollama'},  # any token accepted
        timeout=120
    )
    resp.raise_for_status()
    return resp.json()

result = oai_chat('llama3.2:3b', [{'role': 'user', 'content': 'Say hello.'}])
print('OpenAI-compatible response:')
print(result['choices'][0]['message']['content'])
print()
print('Token usage:', result.get('usage', {}))


### What just happened?

- **`/v1/chat/completions`** mirrors the OpenAI API shape exactly — `choices[0].message.content` is where the reply lives.
- **Any non-empty `Authorization` header** satisfies Ollama's auth check — no real API key required locally.
- **This makes migration trivial:** swap `base_url` from `http://localhost:11434/v1` to `https://api.openai.com/v1` and add a real key to go to production.


In [ ]:
# Challenge: Build a reusable OllamaClient class
#
# Requirements:
#   1. __init__(self, model, base_url, system_prompt='') — store config
#   2. ask(self, question) -> str — single-turn, returns answer text
#   3. chat_session(self) -> list — returns full history of a multi-turn session
#      Use a loop that reads user input until 'quit' is typed
#      (mock: iterate over a pre-defined list of questions instead of input())
#   4. stats(self) -> dict — return {'calls': N, 'total_tokens': N} from internal counters
#
# Your solution here

class OllamaClient:
    def __init__(self, model, base_url='http://localhost:11434', system_prompt=''):
        # TODO: initialise model, base_url, system_prompt, call counter, token counter
        pass

    def ask(self, question):
        # TODO: POST /api/chat with stream=False, return assistant content string
        pass

    def chat_session(self, mock_questions=None):
        # TODO: accumulate history, iterate questions or use input() loop
        pass

    def stats(self):
        # TODO: return call/token counters
        pass

# Smoke-test scaffold:
# client = OllamaClient('llama3.2:3b', system_prompt='You are a helpful assistant.')
# print(client.ask('What is 2 + 2?'))
# print(client.stats())


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Ollama install | Single binary; server auto-starts on port 11434 |
| `ollama pull` | Downloads GGUF model weights to `~/.ollama/models` |
| Modelfile | Layers system prompt + PARAMETER settings on a base model |
| `PARAMETER num_ctx` | Controls KV-cache window; larger = more memory |
| `/api/chat` | Multi-turn endpoint; history is caller-managed, not server state |
| `stream=False` | Collapses streaming into a single JSON response |
| OpenAI compat | `/v1/chat/completions` speaks the OpenAI protocol; easy migration path |
| `eval_count` / `eval_duration` | Built-in throughput metrics in every response |

> **Tip:** Ollama serves on `localhost:11434` by default. The `/api/chat` endpoint uses streaming by default — set `stream=False` in your Python request if you want a single complete response.

---
## What's next
**Day 3** → Quantization — GGUF formats and quality vs speed tradeoffs. Learn how Q4_K_M, Q8_0, and other quantization levels affect file size, RAM usage, and output quality — and when each one is the right choice.

Mark Day 2 complete in your [tracker](../index.html).
